In [5]:
from ultralytics import YOLO
import json
import numpy as np

# Load model
model = YOLO("best_yolov8s.pt")

# Load nutrition database
with open("nutrition_db.json") as f:
    nutrition_db = json.load(f)

print(" Model loaded successfully!")
print(f" Nutrition DB loaded: {len(nutrition_db)} foods")
print(f" Model classes: {len(model.names)}")
print(f" First 5 classes: {list(model.names.values())[:5]}")

 Model loaded successfully!
 Nutrition DB loaded: 103 foods
 Model classes: 26
 First 5 classes: ['french fries', 'biscuit', 'wine', 'coffee', 'milk']


In [7]:
# Test detection on a food image
image_path = r"C:\Users\User\Desktop\food_ai_test\test_image\meal1.png"

results = model(image_path, conf=0.3)[0]

# Show detected foods
print("🍽️ Detected Foods:")
if results.boxes is None or len(results.boxes) == 0:
    print("⚠️ No food detected — try a clearer image")
else:
    for box, conf in zip(results.boxes, results.boxes.conf):
        class_id = int(box.cls.item())
        food_name = results.names[class_id]
        confidence = float(conf.item())
        print(f"  → {food_name:<25} confidence: {confidence:.2f}")

# Show image with detections
results.show()


image 1/1 C:\Users\User\Desktop\food_ai_test\test_image\meal1.png: 480x640 (no detections), 321.6ms
Speed: 6.2ms preprocess, 321.6ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
🍽️ Detected Foods:
⚠️ No food detected — try a clearer image


In [4]:
def estimate_calories(image_path, model, nutrition_db):
    results = model(image_path, conf=0.3)[0]
    
    detected_foods = []
    total_calories = 0
    total_protein = 0
    total_carbs = 0
    total_fat = 0

    if results.masks is None:
        print("⚠️ No masks detected.")
        return []

    for mask, box, conf in zip(
        results.masks.data,
        results.boxes,
        results.boxes.conf
    ):
        class_id = int(box.cls.item())
        food_name = results.names[class_id]
        confidence = float(conf.item())

        # Portion estimation from mask
        mask_np = mask.cpu().numpy()
        portion_ratio = float(mask_np.sum()) / float(mask_np.size)
        estimated_grams = portion_ratio * 500

        # Nutrition lookup
        if food_name in nutrition_db:
            n = nutrition_db[food_name]
            factor = estimated_grams / 100
            calories = n["calories"] * factor
            protein  = n["protein"]  * factor
            carbs    = n["carbs"]    * factor
            fat      = n["fat"]      * factor
        else:
            calories = protein = carbs = fat = 0

        detected_foods.append({
            "food_name":       food_name,
            "confidence":      round(confidence, 2),
            "estimated_grams": round(estimated_grams, 1),
            "calories":        round(calories, 1),
            "protein_g":       round(protein, 1),
            "carbs_g":         round(carbs, 1),
            "fat_g":           round(fat, 1),
        })

        total_calories += calories
        total_protein  += protein
        total_carbs    += carbs
        total_fat      += fat

    # Print results
    print("\n🍽️  Detected Foods:")
    print("-" * 60)
    for food in detected_foods:
        print(f"  🏷️  {food['food_name']:<25} "
              f"~{food['estimated_grams']}g  "
              f"{food['calories']} kcal  "
              f"(conf: {food['confidence']})")
    print("-" * 60)
    print(f"  🔥 Total Calories : {round(total_calories, 1)} kcal")
    print(f"  🥩 Total Protein  : {round(total_protein, 1)} g")
    print(f"  🍞 Total Carbs    : {round(total_carbs, 1)} g")
    print(f"  🧈 Total Fat      : {round(total_fat, 1)} g")

    return detected_foods

# Run it!
results = estimate_calories(
    r"C:\Users\User\Desktop\food_ai_test\test_image\meal1.png",
    model,
    nutrition_db
)


image 1/1 C:\Users\User\Desktop\food_ai_test\test_image\meal1.png: 480x640 1 chicken duck, 1 sauce, 1 potato, 34 cauliflowers, 264.4ms
Speed: 3.2ms preprocess, 264.4ms inference, 48.5ms postprocess per image at shape (1, 3, 480, 640)

🍽️  Detected Foods:
------------------------------------------------------------
  🏷️  cauliflower               ~4.2g  1.1 kcal  (conf: 0.93)
  🏷️  cauliflower               ~17.4g  4.3 kcal  (conf: 0.93)
  🏷️  cauliflower               ~23.7g  5.9 kcal  (conf: 0.92)
  🏷️  sauce                     ~23.0g  20.7 kcal  (conf: 0.91)
  🏷️  cauliflower               ~3.1g  0.8 kcal  (conf: 0.91)
  🏷️  cauliflower               ~5.0g  1.2 kcal  (conf: 0.82)
  🏷️  cauliflower               ~1.2g  0.3 kcal  (conf: 0.82)
  🏷️  cauliflower               ~1.8g  0.4 kcal  (conf: 0.79)
  🏷️  cauliflower               ~0.7g  0.2 kcal  (conf: 0.77)
  🏷️  cauliflower               ~3.1g  0.8 kcal  (conf: 0.73)
  🏷️  cauliflower               ~0.8g  0.2 kcal  (conf: 0.7